## Monte Carlos sur les matrices : Cas des donnees Gaussiennes

L’objectif ici est de comparer les performances de nos estimateurs sur plusieurs matrices de correlations similaires à celles du marché. Ces estimateurs sont :

- L’estimateur empirique
- Le Clipping
- Le Shrinage linéaire 
- Le Shrinage Non linéaire
- L’Oracle RIE (pour le fun)
 
Dans ce cadre, vu que nous ne disposons pas de varies matrices de corrélation des rendements des actions sur le marché (on ne peut avoir que des estimateurs), nous allons prendre des donnees de marché, calculer l’estimateur empirique et celui ci sera considéré comme une vrai matrice de correlation du marché avec laquelle de nouvelles données seront simulées de façon centrée.
On va alors ensuite évaluer nos estimateurs sur ces Nouvelles donnees en faisant une sorte de Monte Carlo (avec simulations gaussiennes) sur l’ensemble des matrices pour chaque ratio de concentration.


In [ ]:
%run setup.py

#### Données

In [ ]:
import datetime
import itertools
import os
import time
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from joblib import Parallel, delayed

from tools.simulation_tools import *
from tools.tools import *


In [ ]:
# ============================================================
# PARAMÈTRES
# ============================================================
ANNEES      = 30      # combien d'années d'historique on veut
N_ACTIONS   = 100     # combien d'actions on veut au final
SEED        = 42      # pour que le tirage aléatoire soit toujours le même
SEUIL_NAN   = 0.0   # on tolère 10% de jours manquants par action
# ============================================================

In [ ]:
# Importation des constituants du sp500

current_dir = Path.cwd().parent # Dossier parent du projet
constituents_path = current_dir / "data" / "raw" / "sp500_constituents.csv"

constituents = pd.read_csv(constituents_path, sep=',')

print(f"{len(constituents)} entreprises trouvées")
print(constituents[["Symbol", "Security", "GICS Sector"]].head())


In [ ]:
end_date = '2026-08-01'
start_date = (datetime(2026, 8, 1) - timedelta(days= ANNEES * 365)).strftime("%Y-%m-%d")  # noqa: DTZ001
start_date

In [ ]:
# Visualisation des données
raw_data_path = current_dir / "data" / "raw" / "sp500_raw_data.csv"

sp500_data = pd.read_csv(raw_data_path)
sp500_data

### Nettoyage des donnees

In [ ]:
# Suppression des actions ayant des jours manquants superieur au seuil
sp500_data_clean = sp500_data.dropna(axis = 1, thresh = (1 - SEUIL_NAN) * len(sp500_data))
sp500_data_clean

In [ ]:
prices = sp500_data_clean

#### Filtre des tickers pour diversification

In [ ]:
# garder, parmi les constituants, que ceux qui ont survecu au nettoyage
survivants = constituents[constituents["Symbol"].isin(prices.columns)].copy()

# un ticker par secteur, tire au hasard
survivants_shuffled = survivants.sample(frac=1, random_state=SEED)
un_par_secteur = (
    survivants_shuffled.groupby("GICS Sector", as_index=False).head(1)
    .sort_values("GICS Sector").reset_index(drop=True)
)
un_par_secteur["Origine"] = "1 par secteur"

# completer avec le reste, tire au hasard sans distinction de secteur
n_secteurs = survivants["GICS Sector"].nunique()
n_reste = N_ACTIONS - n_secteurs
reste_pool = survivants[~survivants["Symbol"].isin(un_par_secteur["Symbol"])]
tirage_complementaire = reste_pool.sample(n=n_reste, random_state=SEED).reset_index(drop=True)
tirage_complementaire["Origine"] = "aléatoire uniforme"

# assembler
selection = pd.concat([un_par_secteur, tirage_complementaire], ignore_index=True)
tickers_finaux = selection["Symbol"].tolist()
prices_final = prices[tickers_finaux]

print(f"\n{len(tickers_finaux)} actions sélectionnées, réparties ainsi :")
print(selection["GICS Sector"].value_counts().sort_index())

In [ ]:
prices_final

#### Derivation des returns

In [ ]:
returns  = prices_final.pct_change().dropna()
returns

In [ ]:
## Un peu de stat 

print(returns.describe().T[["min", "max", "mean", "std"]])

#### Découpage de l'historique en plusieurs blocs temporels

In [ ]:
# Nombre de matrice qu'on veut pour la simu monte carlo
M = 30
p = returns.shape[1]
TAILLE_BLOC = len(returns) // M

print(f"p = {p} actifs")
print(f"gamma_bloc = p/TAILLE_BLOC = {p/TAILLE_BLOC:.3f}")
print(f"{M} blocs de {TAILLE_BLOC} jours chacun ({len(returns) - M*TAILLE_BLOC} jours inutilisés en fin de série)")

blocks = []
True_Sigmas = []
for i in range(M):
    bloc = returns.iloc[i*TAILLE_BLOC : (i+1)*TAILLE_BLOC]
    Xn = normalise(bloc.values)
    S = sample_correlation_matrix(Xn)
    blocks.append(bloc)
    True_Sigmas.append(S)

In [ ]:
for i, S in enumerate(True_Sigmas):
    vals = np.linalg.eigvalsh(S)
    print(f"bloc {i} : plus petite valeur propre = {vals.min():.4f}, "
          f"plus grande = {vals.max():.2f}, trace = {np.trace(S):.1f} (attendu ~{p})")

#### Plage de ratios de concentration

In [ ]:
gamma_values = np.arange(0.1, 3.3, 0.33)
n_values = (p / gamma_values).astype(int)
n_gamma = len(gamma_values)

print(f"{n_gamma} valeurs de gamma, de {gamma_values.min():.2f} à {gamma_values.max():.2f}")
print("n correspondants :", n_values)
print(len(set(n_values)))

#### Boucle sur chaque matrice et calcul des metriques

In [ ]:
estim_names = ["Sample", "Clipping", "Clipping adapted", "Linear", "NLS", "Oracle"]

R = 100  # nombre de repetitions par (bloc, gamma)

erreurs = {nom: np.zeros((M, n_gamma, R)) for nom in estim_names}
trace_ratios = {nom: np.zeros((M, n_gamma, R)) for nom in estim_names}
temps = {nom: np.zeros((M, n_gamma, R)) for nom in estim_names}

In [ ]:
def run_one(i, j, r):
    n = n_values[j]
    q = p / n
    rng = np.random.default_rng(hash((i, j, r)) % (2**32))
    X = rng.multivariate_normal(np.zeros(p), True_Sigmas[i], size=n)
    resultats = mc_matrix_parallel_sim(X, q, n, p, True_Sigmas[i])
    return i, j, r, resultats

R = 100
taches = list(itertools.product(range(M), range(n_gamma), range(R)))
print(f"{len(taches)} simulations a lancer")

sorties = Parallel(n_jobs=-2, verbose=5)(
    delayed(run_one)(i, j, r) for (i, j, r) in taches
)

In [ ]:
for i, j, r, resultats in sorties:
    for nom, (err, tr, t) in resultats.items():
        erreurs[nom][i, j, r] = err
        trace_ratios[nom][i, j, r] = tr
        temps[nom][i, j, r] = t

print("Boucle terminee")

### Visualisation

Ici on passe a la visualation des resutats a travers des graphiques adaptes 
Pour chaque graphique nous essayerons de presenter l'information qu'il renferme et ensuite nous expliquerons les resultats

#### Tableau des moyennes

In [ ]:
moyennes = pd.DataFrame(
    {nom: erreurs[nom].mean(axis=(0, 2)) for nom in estim_names}
).T
moyennes.columns = [f"{g:.2g}" for g in gamma_values]

print("Tableau des moyennes (erreur de Frobenius) :")
moyennes.round(4)

#### Tableau des std

In [ ]:
std_tables = pd.DataFrame(
    {nom: erreurs[nom].std(axis=(0,2)) for nom in estim_names}
).T
std_tables.columns = [f"{g:.2g}" for g in gamma_values]

print("Tableau des ecart-types (erreur de Frobenius) :")
std_tables.round(4)

#### Graphique des moyennes

In [ ]:
couleurs = {
    "Sample": "black",
    "Clipping": "royalblue",
    "Clipping adapted": "darkorange",
    "Linear": "forestgreen",
    "NLS": "red",
    "Oracle": "purple",
}

In [ ]:
plt.figure(figsize=(10, 6))
for nom in estim_names:
    linestyle = "--" if nom == "Oracle" else "-"
    plt.plot(gamma_values, moyennes.loc[nom], marker="o", label=nom, color=couleurs[nom], linestyle=linestyle)
plt.xlabel("Concentration ratio γ = p/n")
plt.ylabel("Erreur de Frobenius (moyenne)")
plt.title("Erreur moyenne par estimateur, en fonction de γ")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#### Graphique des ecart types

In [ ]:
plt.figure(figsize=(10, 6))
for nom in estim_names:
    linestyle = "--" if nom == "Oracle" else "-"
    plt.plot(gamma_values, std_tables.loc[nom], marker="o", label=nom, color=couleurs[nom], linestyle=linestyle)
plt.xlabel("Concentration ratio γ = p/n")
plt.ylabel("Erreur de Frobenius (écart-type)")
plt.title("Écart-type par estimateur, en fonction de γ")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#### Heatmap de rang moyen

Question à laquelle il répond : à chaque niveau de ratio de concentration γ, quel estimateur se classe en moyenne le mieux (et le pire) ?

Insight qu'il permet de sortir : contrairement à une courbe d'erreur brute, le rang est insensible à l'échelle, un estimateur qui explose ponctuellement n'écrase pas la lecture, il obtient juste un mauvais rang sur ce point précis.

In [ ]:
estim_sans_oracle = [nom for nom in estim_names if nom != "Oracle"]

stacked = np.stack([erreurs[nom] for nom in estim_sans_oracle], axis=0)  # (5, M, n_gamma, R)
n_estim = len(estim_sans_oracle)

ranks = np.empty_like(stacked)
for i in range(M):
    for j in range(n_gamma):
        for r in range(stacked.shape[3]):
            order = np.argsort(stacked[:, i, j, r])
            rr = np.empty(n_estim)
            rr[order] = np.arange(1, n_estim + 1)
            ranks[:, i, j, r] = rr

rang_moyen = ranks.mean(axis=(1, 3))  # (5, n_gamma)

plt.figure(figsize=(11, 4.5))
im = plt.imshow(rang_moyen, aspect="auto", cmap="RdYlGn_r", vmin=1, vmax=n_estim)
plt.yticks(range(n_estim), estim_sans_oracle)
plt.xticks(range(len(gamma_values)), [f"{g:.2g}" for g in gamma_values], rotation=45)
plt.xlabel("Concentration ratio γ = p/n")
plt.title("Rang moyen de chaque estimateur (1=meilleur) à chaque γ")
for i in range(n_estim):
    for j in range(len(gamma_values)):
        plt.text(j, i, f"{rang_moyen[i,j]:.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="Rang moyen")
plt.tight_layout()
plt.show()

#### Petits multiples, un panneau par estimateur, échelle log

Question à laquelle il répond : quelle est l'ampleur de l'erreur pour chaque estimateur séparément, avec sa propre échelle, sans qu'aucun n'écrase les autres ?

Insight qu'il permet de sortir : la bande 5-95% (calculée sur M × R observations, pas juste sur R) montre la vraie dispersion : si un estimateur a une bande large à un γ donné alors que les autres restent serrés, c'est un signal d'instabilité localisée, même si sa moyenne semble raisonnable.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True)
for ax, nom in zip(axes.flat, estim_names):
    donnees = erreurs[nom].transpose(1, 0, 2).reshape(len(gamma_values), -1)  # (n_gamma, M*R)
    moyenne_curve = donnees.mean(axis=1)
    p5 = np.percentile(donnees, 5, axis=1)
    p95 = np.percentile(donnees, 95, axis=1)
    ax.plot(gamma_values, moyenne_curve, color=couleurs[nom], linewidth=2)
    ax.fill_between(gamma_values, p5, p95, color=couleurs[nom], alpha=0.2)
    ax.set_yscale("log")
    ax.set_title(nom, fontsize=11)
    ax.grid(alpha=0.3, which="both")
for ax in axes[-1]:
    ax.set_xlabel("γ = p/n")
for ax in axes[:, 0]:
    ax.set_ylabel("Erreur (log)")
plt.tight_layout()
plt.show()

#### Trace ratio : moyenne vs pire cas

Question à laquelle il répond : est-ce que chaque estimateur préserve bien la variance totale (trace ≈ 1), y compris sur son bloc le plus défavorable ?

Insight qu'il permet de sortir : c'est un diagnostic préventif, indépendant de l'erreur de Frobenius. Il peut détecter un effondrement structurel avant même de savoir si l'erreur finale est mauvaise. Le panneau "pire cas" est celui qui compte vraiment : la moyenne peut diluer un problème qui ne touche que 2-3 blocs sur 7.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

tr_mean = np.stack([trace_ratios[nom].mean(axis=(0, 2)) for nom in estim_names])
tr_min = np.stack([trace_ratios[nom].min(axis=(0, 2)) for nom in estim_names])

for ax, data, titre in zip(axes, [tr_mean, tr_min], ["Moyenne (blocs et répétitions)", "Pire cas (minimum)"]):
    im = ax.imshow(data, aspect="auto", cmap="RdYlGn", vmin=0.5, vmax=1.1)
    ax.set_yticks(range(len(estim_names))); ax.set_yticklabels(estim_names)
    ax.set_xticks(range(len(gamma_values))); ax.set_xticklabels([f"{g:.2g}" for g in gamma_values], rotation=45, fontsize=8)
    ax.set_title(titre)
    for i in range(len(estim_names)):
        for j in range(len(gamma_values)):
            ax.text(j, i, f"{data[i,j]:.2f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("trace(estimateur) / trace(True_Sigma)")
plt.tight_layout()
plt.show()

#### Courbe de dépassement de seuil, par tranche de γ

Question à laquelle il répond : à quelle fréquence chaque estimateur échoue-t-il gravement (erreur au-delà d'un certain seuil), et est-ce que ce risque dépend du régime de γ ?

Insight qu'il permet de sortir : une mesure de risque de queue plutôt qu'une moyenne; utile pour argumenter « tel estimateur est fiable en pratique », pas seulement « performant en moyenne ». Le découpage en trois tranches de γ évite de mélanger le régime facile et le régime difficile dans une seule courbe.

In [ ]:
tranches = [("gamma < 0.5", gamma_values < 0.5),
            ("0.5 <= gamma < 1.2", (gamma_values >= 0.5) & (gamma_values < 1.2)),
            ("gamma >= 1.2", gamma_values >= 1.2)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, (titre, mask) in zip(axes, tranches):
    for nom in estim_names:
        donnees = erreurs[nom][:, mask, :].flatten()
        seuils = np.linspace(0, np.percentile(donnees, 99), 100)
        proba_depassement = [(donnees > s).mean() for s in seuils]
        ax.plot(seuils, proba_depassement, color=couleurs[nom], label=nom)
    ax.set_yscale("log")
    ax.set_title(titre)
    ax.set_xlabel("Seuil d'erreur")
    ax.grid(alpha=0.3, which="both")
axes[0].set_ylabel("P(erreur > seuil)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

#### Hétérogénéité entre blocs

Question à laquelle il répond : un problème détecté touche-t-il tous les blocs (donc tous les régimes de marché) de la même façon, ou seulement certaines périodes historiques particulières ?

Insight qu'il permet de sortir : distingue un défaut structurel (lié à γ, présent partout) d'un défaut accidentel (lié à la structure d'un marché précis, par exemple un bloc avec un facteur dominant particulièrement fort).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, nom in zip(axes, ["Linear", "NLS"]):
    grille = erreurs[nom].mean(axis=2)  # moyenne sur R -> (M, n_gamma)
    im = ax.imshow(grille, aspect="auto", cmap="inferno")
    ax.set_xticks(range(len(gamma_values))); ax.set_xticklabels([f"{g:.2g}" for g in gamma_values], rotation=45, fontsize=8)
    ax.set_yticks(range(M)); ax.set_yticklabels([f"bloc {i}" for i in range(M)], fontsize=8)
    ax.set_xlabel("γ = p/n")
    ax.set_title(nom)
    plt.colorbar(im, ax=ax, label="Erreur", fraction=0.046)
plt.tight_layout()
plt.show()

#### Violin plot par tranche de γ 

Question à laquelle il répond : à un γ donné, la distribution des erreurs est-elle homogène, ou bimodale (l'estimateur est soit très bon, soit catastrophique, jamais entre les deux) ?

Insight qu'il permet de sortir : une information invisible dans une moyenne ± écart-type qui révèle un comportement « tout ou rien » qui changerait complètement la façon de faire confiance à l'estimateur.

In [ ]:
tranches = [("gamma < 0.5", gamma_values < 0.5),
            ("0.5 <= gamma < 1.2", (gamma_values >= 0.5) & (gamma_values < 1.2)),
            ("gamma >= 1.2", gamma_values >= 1.2)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=True)
for ax, (titre, mask) in zip(axes, tranches):
    donnees_violin = [erreurs[nom][:, mask, :].flatten() for nom in estim_names]
    parts = ax.violinplot(donnees_violin, showmeans=True, showextrema=True)
    for pc, nom in zip(parts["bodies"], estim_names):
        pc.set_facecolor(couleurs[nom]); pc.set_alpha(0.5)
    ax.set_xticks(range(1, len(estim_names) + 1))
    ax.set_xticklabels(estim_names, rotation=15)
    ax.set_title(titre)
    ax.grid(alpha=0.3, axis="y")
axes[0].set_ylabel("Erreur de Frobenius")
plt.tight_layout()
plt.show()

#### Coût de calcul

Question à laquelle il répond : combien coûte chaque estimateur en temps, indépendamment de sa précision ?

Insight qu'il permet de sortir : une question séparée de la précision (utile pour repérer un coût disproportionné) par rapport au gain.

In [ ]:
moyennes_temps = {nom: temps[nom].mean() * 1000 for nom in estim_names}
ordre_temps = sorted(moyennes_temps, key=moyennes_temps.get)

plt.figure(figsize=(9, 4.5))
plt.barh(ordre_temps, [moyennes_temps[n] for n in ordre_temps],
         color=[couleurs[n] for n in ordre_temps], alpha=0.85)
plt.xscale("log")
plt.xlabel("Temps moyen par simulation (ms, échelle log)")
plt.title("Coût de calcul par estimateur")
plt.grid(alpha=0.3, axis="x", which="both")
plt.tight_layout()
plt.show()

END.